In [ ]:
import os
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Configuration

log_dirs = ["../raw_data/local_5_5", "../raw_data/local_20_20"]
test_runner_log_filename = "test-runner.log"
enforcer_log_filename = "enforcer.log"
asmeta_server_log_filename = "asmeta-server.log"

test_runner_output_filename = "extracted_test_runner_data.csv"
enforcer_output_filename = "extracted_enforcer_data.csv"
asmeta_output_filename = "extracted_asmeta_data.csv"

ping_file = "../raw_data/ping.txt"

log_pattern = re.compile(
    r"Test #(\d+): time: (.*?) ms"
)
enforcer_delay_pattern = re.compile(r"delay\s+([0-9]+(?:\.[0-9]+)?)\s+ms")
asmeta_server_execution_pattern = re.compile(r"Execution time \(in milliseconds\):\s+([0-9]+(?:\.[0-9]+)?)\s+ms")
ping_log_pattern = re.compile(r"time=([\d.]+)\s*ms")

## Enforcement Subsystem data analysis
Computed through data obtained by the Test Runner

In [ ]:
# Data extraction and file saving

records = []

for log_dir in log_dirs:
    experiment_id = os.path.basename(log_dir)
    log_path = os.path.join(log_dir, test_runner_log_filename)

    if not os.path.exists(log_path):
        print(f"File not found: {log_path}")
        continue

    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = log_pattern.search(line)
            if match:
                test_case_number = int(match.group(1))

                time_ms = match.group(2)

                records.append({
                    "experiment": experiment_id,
                    "test_case_number": test_case_number,
                    "delay_ms": time_ms
                })

df_test_runner_raw = pd.DataFrame(records)
df_test_runner_raw["delay_ms"] = pd.to_numeric(df_test_runner_raw["delay_ms"], errors="coerce")

df_test_runner_raw.head()

In [ ]:
# Dataset cleaning

mask = df_test_runner_raw["delay_ms"] < 4
replaced_count = int(mask.sum())
df_test_runner_raw.loc[mask, "delay_ms"] = float("nan")

print(f"Replaced {replaced_count} entries with NaN.")

In [ ]:
if not df_test_runner_raw.empty:
    stats = df_test_runner_raw.groupby("experiment")["delay_ms"].describe()

    stats["variance"] = df_test_runner_raw.groupby("experiment")["delay_ms"].var()
    stats["median"] = df_test_runner_raw.groupby("experiment")["delay_ms"].median()

    print("Stats for the enforcement subsystem delay:")
    display(stats)

    stats.to_csv(test_runner_output_filename)

In [ ]:
# Focus on highest percentiles
percentiles = [0.90, 0.95, 0.99]
pvals = df_test_runner_raw["delay_ms"].dropna().quantile(percentiles)
pvals.index = ["90%", "95%", "99%"]

print("Percentile values (ms):")
display(pvals)

In [ ]:
Q1 = df_test_runner_raw["delay_ms"].dropna().quantile(0.25)
Q2 = df_test_runner_raw["delay_ms"].dropna().quantile(0.5)
Q3 = df_test_runner_raw["delay_ms"].dropna().quantile(0.75)

bowley_skewness = (Q3 + Q1 - 2*Q2) / (Q3 - Q1)

print(f"Bowley skewness: {bowley_skewness:.3f}")

## Enforcer data analysis

In [ ]:
enforcer_iteration_records = []

for log_dir in log_dirs:
    experiment = os.path.basename(log_dir)
    log_path = os.path.join(log_dir, enforcer_log_filename)

    if not os.path.exists(log_path):
        print(f"File not found: {log_path}")
        continue

    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = enforcer_delay_pattern.search(line)
            if match:
                delay_value = float(match.group(1))
                enforcer_iteration_records.append({
                    "experiment": experiment,
                    "delay_ms": delay_value
                })

df_enforcer_raw = pd.DataFrame(enforcer_iteration_records)

if not df_enforcer_raw.empty:
    stats = df_enforcer_raw.groupby("experiment")["delay_ms"].describe()

    stats["variance"] = df_enforcer_raw.groupby("experiment")["delay_ms"].var()
    stats["median"] = df_enforcer_raw.groupby("experiment")["delay_ms"].median()

    print("Stats for the enforcer delay:")
    display(stats)

    stats.to_csv(enforcer_output_filename)

In [ ]:
# Focus on highest percentiles
percentiles = [0.90, 0.95, 0.99]
pvals = df_enforcer_raw["delay_ms"].dropna().quantile(percentiles)
pvals.index = ["90%", "95%", "99%"]

print("Percentile values (ms):")
display(pvals)

skew_value = df_enforcer_raw['delay_ms'].skew()
print(f"Skewness: {skew_value:.3f}")

Q1 = df_enforcer_raw["delay_ms"].dropna().quantile(0.25)
Q2 = df_enforcer_raw["delay_ms"].dropna().quantile(0.5)
Q3 = df_enforcer_raw["delay_ms"].dropna().quantile(0.75)

bowley_skewness = (Q3 + Q1 - 2*Q2) / (Q3 - Q1)

print(f"Bowley skewness: {bowley_skewness:.3f}")

## ASMETA Server data analysis

In [ ]:
asmeta_enforcement_records = []

for log_dir in log_dirs:
    experiment = os.path.basename(log_dir)
    log_path = os.path.join(log_dir, asmeta_server_log_filename)

    if not os.path.exists(log_path):
        print(f"File not found: {log_path}")
        continue

    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            match = asmeta_server_execution_pattern.search(line)
            if match:
                delay_value = float(match.group(1))
                asmeta_enforcement_records.append({
                    "experiment": experiment,
                    "delay_ms": delay_value
                })

df_asmeta_raw = pd.DataFrame(asmeta_enforcement_records)

if not df_asmeta_raw.empty:
    stats = df_asmeta_raw.groupby("experiment")["delay_ms"].describe()

    stats["variance"] = df_asmeta_raw.groupby("experiment")["delay_ms"].var()
    stats["median"] = df_asmeta_raw.groupby("experiment")["delay_ms"].median()

    # Overall (all data without grouping)
    overall = df_asmeta_raw["delay_ms"].describe()
    overall["variance"] = df_asmeta_raw["delay_ms"].var()
    overall["median"] = df_asmeta_raw["delay_ms"].median()
    stats.loc["all"] = overall

    print("Stats for the asmeta delay:")
    display(stats)

    stats.to_csv(asmeta_output_filename)

In [ ]:
# Focus on highest percentiles
percentiles = [0.90, 0.95, 0.99]
pvals = df_asmeta_raw["delay_ms"].dropna().quantile(percentiles)
pvals.index = ["90%", "95%", "99%"]

print("Percentile values (ms):")
display(pvals)

Q1 = df_asmeta_raw["delay_ms"].dropna().quantile(0.25)
Q2 = df_asmeta_raw["delay_ms"].dropna().quantile(0.5)
Q3 = df_asmeta_raw["delay_ms"].dropna().quantile(0.75)

bowley_skewness = (Q3 + Q1 - 2*Q2) / (Q3 - Q1)

print(f"Bowley skewness: {bowley_skewness:.3f}")

## Data reporting

In [ ]:
df_overall_mean = pd.concat(
    [df_test_runner_raw.groupby("experiment")["delay_ms"].mean().rename("enforcement_subsystem"),
     df_enforcer_raw.groupby("experiment")["delay_ms"].mean().rename("enforcer"),
     df_asmeta_raw.groupby("experiment")["delay_ms"].mean().rename("asmeta_server")],
    axis=1
).reset_index()

df_overall_mean["rules"] = df_overall_mean["experiment"].apply(lambda x: int(re.search(r'_(\d+)_(\d+)$', x).group(1)))
df_overall_mean["clauses_per_rule"] = df_overall_mean["experiment"].apply(lambda x: int(re.search(r'_(\d+)_(\d+)$', x).group(2)))
df_overall_mean["conditions_checked"] = df_overall_mean["rules"] * df_overall_mean["clauses_per_rule"]

df_overall_mean = df_overall_mean[["experiment", "rules", "clauses_per_rule", "conditions_checked", "enforcement_subsystem", "enforcer", "asmeta_server"]].sort_values(by=["conditions_checked"])

df_overall_mean

In [ ]:
df_bar_plot = df_overall_mean.set_index("experiment")[["asmeta_server", "enforcer", "enforcement_subsystem"]]
labels = ["ASMETA server", "Enforcer", "Enforcement Subsystem"]

x = np.arange(len(df_bar_plot.index))
width = 0.25

fig, ax = plt.subplots(figsize=(6, 4))
palette = sns.color_palette("Set2", 3)
palette.reverse()

for i, col in enumerate(df_bar_plot.columns):
    ax.bar(x + (i - 1) * width, df_bar_plot[col].values, width, label=labels[i], color=palette[i])

ax.set_xticks(x)
ax.set_xticklabels(df_bar_plot.index, rotation=0)
ax.set_ylabel("Mean Overhead (ms)")
ax.set_xlabel("")
ax.legend(title="Component")

plt.tight_layout()
plt.show()


In [ ]:
df_xy_plot = df_overall_mean.set_index("conditions_checked")[["asmeta_server", "enforcer", "enforcement_subsystem"]]

markers = ['o', 's', '^']
labels = ["ASMETA server", "Enforcer", "Enforcement Subsystem"]
palette = sns.color_palette("Set2", 3)
palette.reverse()

fig, ax = plt.subplots(figsize=(6, 4))

x = df_xy_plot.index.values
for col, marker, label, color in zip(df_xy_plot.columns, markers, labels, palette):
    y = df_xy_plot[col].values
    ax.plot(x, y, marker=marker, linestyle='-', markersize=8, color=color, label=label)

ax.set_xticks(x)
ax.set_xticklabels(x)
ax.set_xlabel("Conditions checked")
ax.set_ylabel("Mean Execution Time (ms)")
ax.legend(title="Component")
plt.tight_layout()
plt.show()

## Ping analysis

In [ ]:
ping_path = os.path.join(ping_file)

if not os.path.exists(ping_path):
    print(f"File not found: {ping_path}")

else:
    ping_records = []
    with open(ping_path, "r", encoding="utf-8") as f:
        for line in f:
            match = ping_log_pattern.search(line)
            if match:
                time_value = float(match.group(1))
                ping_records.append({
                    "time": time_value
                })

    df_ping = pd.DataFrame(ping_records)

    if not df_ping.empty:
        stats = df_ping["time"].describe()

        stats["variance"] = df_ping["time"].var()
        stats["median"] = df_ping["time"].median()

        print("Ping data stats:")
        display(stats)